## Full Data Waterfall Plot

In [1]:
print('Loading Packages...\n')
from lib.imports import *
from lib.constants import *
from lib.reco_functions import (
    reco_energy,
    bunch_cutting,
    pairwise_relative_direction,
    is_inside_rotated_parabola,
)

# # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # #
# Analyze Base GENIE files ('Base') or Empirical Model ('Empirical')

which_MC, POT = select_MC()
print('\n', POT, 'POT')

# REQUIRES USER INPUT (specify path to MC files)
MC_directory = '../MC_GENIE-Base/'   # or 'MC_GENIE-Base/'
plots_dir = 'plots/'                # directory to hold plots
os.system('mkdir -p ' + plots_dir)  # auto-generate plots directory if not present

print('\nWill use ' + MC_directory + ' for the MC samples, and ' + str(plots_dir) +' for the plots output directory')

data_directory = '../../FY22_23/'
csv_file = 'lib/POT_trigger_FY22_23_summary.csv'   # contains trigger counts and POT for on-beam data
offbeam_directory = '../../off_beam/'
csv_file_ext = 'lib/offbeam_summary.csv'           # contains trigger counts for off-beam data

# simulated skyshine files
skyshine_file_names = [
    '../../skyshine_100k_no_ext_clusters.root',
    '../../skyshine_100k_no_ext_clusters_2.root',
    '../../skyshine_100k_no_ext_clusters_3.root'
]

EXPECTED_SKYSHINE = 14.1            # number of skyshine events expcted in our signal region, determined from the sideband analysis

print('\ndone')

Loading Packages...




Please specify which MC sample:
  1 = Empirical model
  2 = Base GENIE
Enter your selection (1 or 2):  1



******* Empirical Model Selected *******


 4.41733e+20 POT

Will use ../MC_GENIE-Base/ for the MC samples, and plots/ for the plots output directory

done


In [2]:
# Truth-level "no selection" counts, for ALL 6 categories (not just signal).
# Loops the RAW trigger-tree branches per file (not the cluster-expanded
# versions from the loader cell below, which would double-count events that
# produce more than one cluster). Feeds the "True (no selection)" row of the
# waterfall table and the efficiency denominator.

MC_TRUE_by_cat     = np.zeros(6)   # weighted (GENIE CV weight), indices: 0=nuNCQE,1=NCother,2=CC,3=nubarNCQE,4=OOFV,5=External
MC_TRUE_raw_by_cat = np.zeros(6, dtype=int)

file_names = [f for f in os.listdir(MC_directory) if os.path.isfile(os.path.join(MC_directory, f))]

for file_idx in trange(len(file_names), desc='Loading MC files (truth pass)'):
    path = os.path.join(MC_directory, file_names[file_idx])
    root = uproot.open(path)

    S   = root['phaseIITriggerTree']
    vx  = S['trueNuIntxVtx_X'].array(library='np')
    vy  = S['trueNuIntxVtx_Y'].array(library='np')
    vz  = S['trueNuIntxVtx_Z'].array(library='np')
    nc  = S['trueNC'].array(library='np')
    qel = S['trueQEL'].array(library='np')
    nu  = S['trueFSLPdg'].array(library='np')
    tz  = S['trueTargetZ'].array(library='np')
    wt  = S['weight_TunedCentralValue_UBGenie'].array(library='np')

    vx_cm = vx / 100
    vy_cm = vy / 100 + 0.1446
    vz_cm = vz / 100 - 1.681
    r2    = vx_cm**2 + vz_cm**2

    ext_flags = (r2 > radius**2) | (np.abs(vy_cm) > half_height)      # true vertex outside the tank -> External/dirt
    fv_flags  = (r2 < FV_radius**2) & (np.abs(vy_cm) < FV_half_height) # true vertex inside the FV

    is_nc    = (nc == 1)
    is_qel   = (qel == 1)
    is_O16   = (tz == 8)
    is_nu    = (nu == 14) | (nu == 12)
    is_nubar = (nu == -14) | (nu == -12)

    # same priority order as the reco-level categorization below:
    # External -> OOFV -> CC -> NCother -> nuNCQE / nubarNCQE
    cat = np.full(len(vx_cm), 1, dtype=int)   # default bucket = NCother (see fallback note)
    cat[ext_flags] = 5

    in_tank = ~ext_flags
    cat[in_tank & ~fv_flags] = 4

    in_fv = in_tank & fv_flags
    cat[in_fv & ~is_nc] = 2

    nc_fv = in_fv & is_nc
    cat[nc_fv & (~is_qel | ~is_O16)] = 1   # NC but not QEL, or QEL but not on O16 -> NCother

    qel_o16 = nc_fv & is_qel & is_O16
    cat[qel_o16 & is_nu]    = 0
    cat[qel_o16 & is_nubar] = 3
    # qel_o16 & (~is_nu & ~is_nubar): exotic trueFSLPdg (not std. BNB content) -> stays NCother(1),
    # flagged below rather than silently mis-binned

    fallback = qel_o16 & ~is_nu & ~is_nubar
    if fallback.any():
        print(f'  [warn] {file_names[file_idx]}: {int(fallback.sum())} QEL-on-O16 events with '
              f'unexpected trueFSLPdg -> bucketed as NCother')

    wt_cv = np.concatenate(wt).astype(float)
    wt_cv = np.where(np.isinf(wt_cv), 1.0, wt_cv)   # match the inf-weight handling used at reco level

    for c in range(6):
        m = (cat == c)
        MC_TRUE_by_cat[c]     += wt_cv[m].sum()
        MC_TRUE_raw_by_cat[c] += int(m.sum())

cat_names_ordered = ['nuNCQE', 'NCother', 'CC', 'nubarNCQE', 'OOFV', 'External']
print('\nTrue (no selection) MC counts, raw MC POT:')
for name, w, r in zip(cat_names_ordered, MC_TRUE_by_cat, MC_TRUE_raw_by_cat):
    print(f'  {name:10s}  weighted={w:12.2f}   raw={r}')

# kept for continuity with earlier variable names used elsewhere in the notebook
total_weighted     = MC_TRUE_by_cat[0]
NCQE_signal_events = total_weighted
print(f'\nTotal true NCQE signal (weighted): {total_weighted:.4f}')
print('done')

Loading MC files (truth pass): 100%|██████████████████████████████████████████████████████████████████████████| 3930/3930 [01:30<00:00, 43.29it/s]


True (no selection) MC counts, raw MC POT:
  nuNCQE      weighted=    19553.00   raw=19553
  NCother     weighted=    22427.00   raw=22427
  CC          weighted=   114970.43   raw=97879
  nubarNCQE   weighted=      349.00   raw=349
  OOFV        weighted=  1184485.44   raw=1056005
  External    weighted=  4907390.43   raw=4332821

Total true NCQE signal (weighted): 19553.0000
done


In [3]:
# Load MC samples

file_names = [f for f in os.listdir(MC_directory) if os.path.isfile(os.path.join(MC_directory, f))]

# scalar accumulators
MC_cluster_number    = []
MC_cluster_time      = []
MC_cluster_charge    = []
MC_cluster_Qb        = []
MC_cluster_Hits      = []
MC_TankMRDCoinc      = []
MC_NoVeto            = []
MC_recoVtx           = []
MC_recoVty           = []
MC_recoVtz           = []
MC_Vtx               = []
MC_Vty               = []
MC_Vtz               = []
MC_Vtt               = []
MC_bunchTimes        = []
MC_External          = []
MC_FV                = []
MC_NC                = []
MC_QEL               = []
MC_MEC               = []
MC_PDG               = []
MC_Z                 = []
MC_CV_weight         = []

# jagged accumulators
MC_hitX  = []
MC_hitY  = []
MC_hitZ  = []
MC_hitT  = []
MC_hitPE = []
MC_MRD_hitT  = []
MC_FMV_hitT  = []

for file_idx in trange(len(file_names), desc='Loading MC files'):
    path = os.path.join(MC_directory, file_names[file_idx])
    root = uproot.open(path)

    # ── cluster tree ──────────────────────────────────────────────────────────
    T   = root['phaseIITankClusterTree']
    CEN = T['eventNumber'].array(library='np')

    MC_cluster_number.append(T['clusterNumber'].array(library='np'))
    MC_cluster_charge.append(T['clusterPE'].array(library='np'))
    MC_cluster_Qb.append(T['clusterChargeBalance'].array(library='np'))
    MC_cluster_Hits.append(T['clusterHits'].array(library='np'))
    MC_cluster_time.append(T['clusterTime'].array(library='np'))
    MC_bunchTimes.append(T['bunchTimes'].array(library='np'))
    MC_TankMRDCoinc.append(T['TankMRDCoinc'].array(library='np'))
    MC_NoVeto.append(T['NoVeto'].array(library='np'))
    MC_recoVtx.append(T['recoLeastSqVtxX'].array(library='np'))
    MC_recoVty.append(T['recoLeastSqVtxY'].array(library='np'))
    MC_recoVtz.append(T['recoLeastSqVtxZ'].array(library='np'))

    MC_hitX.append(T['hitX'].array())
    MC_hitY.append(T['hitY'].array())
    MC_hitZ.append(T['hitZ'].array())
    MC_hitT.append(T['hitT'].array())
    MC_hitPE.append(T['hitPE'].array())

    # ── trigger tree ──────────────────────────────────────────────────────────
    S   = root['phaseIITriggerTree']
    vx  = S['trueNuIntxVtx_X'].array(library='np')
    vy  = S['trueNuIntxVtx_Y'].array(library='np')
    vz  = S['trueNuIntxVtx_Z'].array(library='np')

    vx_cm = vx / 100
    vy_cm = vy / 100 + 0.1446
    vz_cm = vz / 100 - 1.681
    r2    = vx_cm**2 + vz_cm**2
    ext_flags = (r2 > radius**2)    | (np.abs(vy_cm) > half_height)
    fv_flags  = (r2 < FV_radius**2) & (np.abs(vy_cm) < FV_half_height)

    # map trigger-level quantities onto clusters via CEN
    MC_Vtx.append(vx_cm[CEN])
    MC_Vty.append(vy_cm[CEN])
    MC_Vtz.append(vz_cm[CEN])
    MC_Vtt.append(S['trueNuIntxVtx_T'].array(library='np')[CEN])
    MC_External.append(ext_flags[CEN])
    MC_FV.append(fv_flags[CEN])
    MC_NC.append(S['trueNC'].array(library='np')[CEN])
    MC_QEL.append(S['trueQEL'].array(library='np')[CEN])
    MC_MEC.append(S['trueMEC'].array(library='np')[CEN])
    MC_PDG.append(S['trueFSLPdg'].array(library='np')[CEN])
    MC_Z.append(S['trueTargetZ'].array(library='np')[CEN])
    
    MC_CV_weight.append(S['weight_TunedCentralValue_UBGenie'].array(library='np')[CEN])

    mrd_t   = S['MRDhitT'].array()
    veto_t  = S['FMVhitT'].array()
    MC_MRD_hitT.append(mrd_t[CEN])
    MC_FMV_hitT.append(veto_t[CEN])

# ── concatenate everything once ───────────────────────────────────────────────
print('\nManaging arrays...\n')

MC_cluster_number  = np.concatenate(MC_cluster_number)
MC_cluster_time    = np.concatenate(MC_cluster_time)
MC_cluster_charge  = np.concatenate(MC_cluster_charge)
MC_cluster_Qb      = np.concatenate(MC_cluster_Qb)
MC_cluster_Hits    = np.concatenate(MC_cluster_Hits)
MC_bunchTimes      = np.concatenate(MC_bunchTimes)
MC_TankMRDCoinc    = np.concatenate(MC_TankMRDCoinc)
MC_NoVeto          = np.concatenate(MC_NoVeto)
MC_recoVtx         = np.concatenate(MC_recoVtx)
MC_recoVty         = np.concatenate(MC_recoVty)
MC_recoVtz         = np.concatenate(MC_recoVtz)
MC_Vtx             = np.concatenate(MC_Vtx)
MC_Vty             = np.concatenate(MC_Vty)
MC_Vtz             = np.concatenate(MC_Vtz)
MC_Vtt             = np.concatenate(MC_Vtt)
MC_External        = np.concatenate(MC_External)
MC_FV              = np.concatenate(MC_FV)
MC_NC              = np.concatenate(MC_NC)
MC_QEL             = np.concatenate(MC_QEL)
MC_MEC             = np.concatenate(MC_MEC)
MC_PDG             = np.concatenate(MC_PDG)
MC_Z               = np.concatenate(MC_Z)
MC_CV_weight       = np.concatenate(MC_CV_weight)

MC_hitX     = ak.concatenate(MC_hitX)
MC_hitY     = ak.concatenate(MC_hitY)
MC_hitZ     = ak.concatenate(MC_hitZ)
MC_hitT     = ak.concatenate(MC_hitT)
MC_hitPE    = ak.concatenate(MC_hitPE)
MC_MRD_hitT = ak.concatenate(MC_MRD_hitT)
MC_FMV_hitT = ak.concatenate(MC_FMV_hitT)

print(f'POT: {POT:.3e}')
print(f'Total clusters: {len(MC_cluster_number)}')
print('\ndone')

Loading MC files: 100%|███████████████████████████████████████████████████████████████████████████████████████| 3930/3930 [03:07<00:00, 20.95it/s]



Managing arrays...

POT: 4.426e+20
Total clusters: 3299463

done


In [4]:
# on-beam data (can be found here: /pnfs/annie/persistent/users/doran/datasets/NCQE_BEAMCLUSTER_DATA/EXTRACTED_FULL_BEAM_DATA/)

def extract_run_number(file_name):
    return int(file_name.split('_')[0][1:])  # Split by '_' and remove 'R' to get the run number

file_names = [file for file in os.listdir(data_directory) if os.path.isfile(os.path.join(data_directory, file))]
print('There are: ', len(file_names), ' files\n')

# Extract all run numbers from the current files
run_numbers = [extract_run_number(f) for f in file_names]

# POT file
full_pot_df = pd.read_csv(csv_file)

# Filter the dataframe to only include runs that are in the <STRING>/ directory
pot_df = full_pot_df[full_pot_df['RUN'].isin(run_numbers)]

# extract total POT and triggers for only the matched runs
total_POT_TOR860 = sum(pot_df["TOR860_POTe12"])/(1e8)   # convert to e20 POT
total_triggers = sum(pot_df["TOTAL_TRIGGERS"])

cluster_time = []
cluster_charge = []
cluster_Hits = []
cluster_Qb = []
TankMRDCoinc = []
NoVeto = []
only_prompt_cluster = []
MRD_activity = []
Veto_activity = []
Vtx_reco = []
Vty_reco = []
Vtz_reco = []
hitX = []
hitY = []
hitZ = []
hitT = []
hitPE = []
runs = []

counter = 0
print('(', (counter), '/', len(file_names), ')')

for file_name in file_names:
    
    run_number = extract_run_number(file_name)
    counter += 1
    
    if counter % 50 == 0:
        print('(', (counter), '/', len(file_names), ')')
    
    with uproot.open(os.path.join(data_directory, file_name)) as file:
        
        Event = file["data"]
        
        runs.append(Event["run_number"].array(library="np"))
        
        # needed for both
        cluster_time.append(Event["cluster_time_BRF"].array(library="np"))
        cluster_charge.append(Event["cluster_PE"].array(library="np"))
        cluster_Qb.append(Event["cluster_Qb"].array(library="np"))
        cluster_Hits.append(Event["cluster_Hits"].array(library="np"))
        TankMRDCoinc.append(Event["TankMRDCoinc"].array(library="np"))
        NoVeto.append(Event["NoVeto"].array(library="np"))
        only_prompt_cluster.append(Event["only_prompt_cluster"].array(library="np"))
        MRD_activity.append(Event["MRD_activity"].array(library="np"))
        Veto_activity.append(Event["FMV_activity"].array(library="np"))
        Vtx_reco.append(Event["recoVtx"].array(library="np"))
        Vty_reco.append(Event["recoVty"].array(library="np"))
        Vtz_reco.append(Event["recoVtz"].array(library="np"))
        
        hitX.append(Event["hitX"].array())
        hitY.append(Event["hitY"].array())
        hitZ.append(Event["hitZ"].array())
        hitT.append(Event["hitT"].array())
        hitPE.append(Event["hitPE"].array())

        #print(len(cluster_time[-1]), "clusters")
        
        
# Concatenate everything once at the end
print('\nManaging arrays...\n')
runs = np.concatenate(runs)
cluster_time = np.concatenate(cluster_time)
cluster_charge = np.concatenate(cluster_charge)
cluster_Qb = np.concatenate(cluster_Qb)
cluster_Hits = np.concatenate(cluster_Hits)
TankMRDCoinc = np.concatenate(TankMRDCoinc)
NoVeto = np.concatenate(NoVeto)
only_prompt_cluster = np.concatenate(only_prompt_cluster)
MRD_activity = np.concatenate(MRD_activity)
Veto_activity = np.concatenate(Veto_activity)
Vtx_reco = np.concatenate(Vtx_reco)
Vty_reco = np.concatenate(Vty_reco)
Vtz_reco = np.concatenate(Vtz_reco)

# Jagged arrays
hitX = ak.concatenate(hitX)
hitY = ak.concatenate(hitY)
hitZ = ak.concatenate(hitZ)
hitT = ak.concatenate(hitT)
hitPE = ak.concatenate(hitPE)

print(total_triggers, 'triggers')
print(total_POT_TOR860, 'e20 POT\n')

There are:  389  files

( 0 / 389 )
( 50 / 389 )
( 100 / 389 )
( 150 / 389 )
( 200 / 389 )
( 250 / 389 )
( 300 / 389 )
( 350 / 389 )

Managing arrays...

60624559 triggers
2.572970591694 e20 POT



In [5]:
# off beam data (can be found here: /pnfs/annie/persistent/users/doran/datasets/NCQE_BEAMCLUSTER_DATA/EXTRACTED_FULL_OFFBEAM_DATA/)
file_names_ext = [file for file in os.listdir(offbeam_directory) if os.path.isfile(os.path.join(offbeam_directory, file))]
print('There are: ', len(file_names_ext), ' files\n')

# POT file
pot_df_ext = pd.read_csv(csv_file_ext)

# extract total POT and triggers
total_ext_triggers = sum(pot_df_ext["TOTAL_TRIGGERS"])

cluster_time_ext = []
cluster_charge_ext = []
cluster_Hits_ext = []
cluster_Qb_ext = []
TankMRDCoinc_ext = []
NoVeto_ext = []
only_prompt_cluster_ext = []
MRD_activity_ext = []
Veto_activity_ext = []
Vtx_reco_ext = []
Vty_reco_ext = []
Vtz_reco_ext = []
hitX_ext = []
hitY_ext = []
hitZ_ext = []
hitT_ext = []
hitPE_ext = []
runs_ext = []

counter = 0
print('(', (counter), '/', len(file_names_ext), ')')

for file_name in file_names_ext:
    
    run_number = extract_run_number(file_name)
    counter += 1
    
    if counter % 50 == 0:
        print('(', (counter), '/', len(file_names_ext), ')')
    
    with uproot.open(os.path.join(offbeam_directory, file_name)) as file:
        
        Event = file["data"]
        
        runs_ext.append(Event["run_number"].array(library="np"))
        
        # needed for both
        cluster_time_ext.append(Event["cluster_time"].array(library="np"))
        cluster_charge_ext.append(Event["cluster_PE"].array(library="np"))
        cluster_Qb_ext.append(Event["cluster_Qb"].array(library="np"))
        cluster_Hits_ext.append(Event["cluster_Hits"].array(library="np"))
        TankMRDCoinc_ext.append(Event["TankMRDCoinc"].array(library="np"))
        NoVeto_ext.append(Event["NoVeto"].array(library="np"))
        only_prompt_cluster_ext.append(Event["only_prompt_cluster"].array(library="np"))
        MRD_activity_ext.append(Event["MRD_activity"].array(library="np"))
        Veto_activity_ext.append(Event["FMV_activity"].array(library="np"))
        Vtx_reco_ext.append(Event["recoVtx"].array(library="np"))
        Vty_reco_ext.append(Event["recoVty"].array(library="np"))
        Vtz_reco_ext.append(Event["recoVtz"].array(library="np"))
        
        hitX_ext.append(Event["hitX"].array())
        hitY_ext.append(Event["hitY"].array())
        hitZ_ext.append(Event["hitZ"].array())
        hitT_ext.append(Event["hitT"].array())
        hitPE_ext.append(Event["hitPE"].array())
        
        
# Concatenate everything once at the end
print('\nManaging arrays...\n')
runs_ext = np.concatenate(runs_ext)
cluster_time_ext = np.concatenate(cluster_time_ext)
cluster_charge_ext = np.concatenate(cluster_charge_ext)
cluster_Qb_ext = np.concatenate(cluster_Qb_ext)
cluster_Hits_ext = np.concatenate(cluster_Hits_ext)
TankMRDCoinc_ext = np.concatenate(TankMRDCoinc_ext)
NoVeto_ext = np.concatenate(NoVeto_ext)
only_prompt_cluster_ext = np.concatenate(only_prompt_cluster_ext)
MRD_activity_ext = np.concatenate(MRD_activity_ext)
Veto_activity_ext = np.concatenate(Veto_activity_ext)
Vtx_reco_ext = np.concatenate(Vtx_reco_ext)
Vty_reco_ext = np.concatenate(Vty_reco_ext)
Vtz_reco_ext = np.concatenate(Vtz_reco_ext)

# Jagged arrays
hitX_ext = ak.concatenate(hitX_ext)
hitY_ext = ak.concatenate(hitY_ext)
hitZ_ext = ak.concatenate(hitZ_ext)
hitT_ext = ak.concatenate(hitT_ext)
hitPE_ext = ak.concatenate(hitPE_ext)

print(total_ext_triggers, 'triggers')

There are:  389  files

( 0 / 389 )
( 50 / 389 )
( 100 / 389 )
( 150 / 389 )
( 200 / 389 )
( 250 / 389 )
( 300 / 389 )
( 350 / 389 )

Managing arrays...

24737351 triggers


In [6]:
# MC waterfall: applies the 10 selection cuts in the 
# ORDER YOUR TABLE REPORTS THEM (Clusterization, No FMV/MRD, In Spill, Only Prompt, q_b, FV, E_reco,
# 1-sigma BNB, C_V, 1/3 spill), not the order 05_FULL_DATA_signal_band.ipynb
# currently executes them in (which runs 1/3-spill early and 1-sigma BNB
# last, purely as a computational shortcut). Because every stage is an AND
# of independent boolean cuts, the final surviving set is unaffected by this
# reordering -- only the intermediate cascade counts change.

N_CAT = 6
STAGE_NAMES = ['Clusterization', 'No FMV/MRD', 'In Spill', 'Only Prompt', 'q_b',
               'FV', 'E_reco', '1sigma BNB', 'C_V', '1/3 spill']

MC_waterfall = np.zeros((len(STAGE_NAMES), N_CAT))   # weighted cumulative counts [stage, category]

def categorize_mc(i):
    """Same decision tree as the truth-level pass, applied to the cluster-indexed
    truth arrays (MC_External[i] etc. carry the parent event's truth info)."""
    if MC_External[i]:
        return 5
    if not MC_FV[i]:
        return 4
    if MC_NC[i] != 1:
        return 2
    if MC_QEL[i] == 1 and MC_Z[i] == 8:
        if MC_PDG[i] == 14 or MC_PDG[i] == 12:
            return 0
        if MC_PDG[i] == -14 or MC_PDG[i] == -12:
            return 3
        return 1   # exotic FSL PDG on O16 QEL -- see truth-level cell note
    return 1        # NC, not QEL-on-O16 -> NCother


for i in trange(len(MC_cluster_time), desc='MC waterfall'):

    weight = MC_CV_weight[i][0]
    if np.isinf(weight):
        weight = 1.0

    indy = categorize_mc(i)

    # 1. Clusterization
    c1 = (MC_cluster_number[i] == 0 and MC_bunchTimes[i] <= time_to_prompt_end
          and MC_cluster_Hits[i] >= 10)
    if not c1:
        continue
    MC_waterfall[0, indy] += weight

    # 2. No FMV/MRD
    c2 = not (
        MC_TankMRDCoinc[i] == 1
        or MC_NoVeto[i] == 0
        or any(0 <= t <= 300 for t in MC_MRD_hitT[i])
        or any(0 <= t <= 300 for t in MC_FMV_hitT[i])
    )
    if not c2:
        continue
    MC_waterfall[1, indy] += weight

    # 3. In Spill
    c3 = bunch_time_start <= MC_bunchTimes[i] <= bunch_time_end
    if not c3:
        continue
    MC_waterfall[2, indy] += weight

    # 4. Only Prompt (no secondary prompt cluster later in this event)
    skip_event = False
    for k in range(i + 1, len(MC_cluster_time)):
        if MC_cluster_number[k] != 0:
            if MC_bunchTimes[k] < time_to_prompt_end:
                skip_event = True
            break
        else:
            break
    c4 = not skip_event
    if not c4:
        continue
    MC_waterfall[3, indy] += weight

    # 5. q_b
    c5 = MC_cluster_Qb[i] < 0.6
    if not c5:
        continue
    MC_waterfall[4, indy] += weight

    # 6. FV
    r2 = MC_recoVtx[i]**2 + MC_recoVtz[i]**2
    c6 = (MC_recoVtx[i] >= -50) and (r2 <= FV_radius**2) and (np.abs(MC_recoVty[i]) <= FV_half_height)
    if not c6:
        continue
    MC_waterfall[5, indy] += weight

    # 7. E_reco
    recoE = reco_energy(MC_cluster_charge[i])
    c7 = 5 <= recoE <= 12
    if not c7:
        continue
    MC_waterfall[6, indy] += weight

    # 8. 1-sigma BNB (per-bunch timing)
    c8 = bunch_cutting(MC_bunchTimes[i], centers)
    if c8:
        MC_waterfall[7, indy] += weight

    # 9. C_V (pairwise-direction / parabola cut) -- computed regardless of c8,
    # same as the original notebook (needed for both signal & outside buckets)
    cw = pairwise_relative_direction(MC_hitX[i], MC_hitY[i], MC_hitZ[i], MC_hitPE[i], MC_hitT[i])
    c9 = is_inside_rotated_parabola(cw[2], cw[0])
    if c8 and c9:
        MC_waterfall[8, indy] += weight

    # 10. 1/3 spill (final)
    c10 = MC_bunchTimes[i] <= MC_third_spill_cut
    if c8 and c9 and c10:
        MC_waterfall[9, indy] += weight


print('done')

MC waterfall: 100%|██████████████████████████████████████████████████████████████████████████████████| 3299463/3299463 [01:01<00:00, 53253.80it/s]

done


In [7]:
# on-beam data

# On-beam waterfall, same table-order cascade as the MC cell.
#
# IMPORTANT LIMITATION: the extracted data ntuples only expose a single
# `only_prompt_cluster` boolean, which (per the original notebook's own
# comment) covers "CN==0 and cluster is in the prompt window" -- there is no
# separate field to isolate "no secondary prompt cluster in the event" the
# way the MC cluster-tree loop above does. So for data, "Clusterization" and
# "Only Prompt" are NECESSARILY identical counts here (both gated by the same
# flag) -- this isn't a bug, it's a granularity limit of the current ntuple
# extraction. Revisit upstream if you need these separated for data.

waterfall_onbeam = np.zeros(len(STAGE_NAMES))

for i in trange(len(cluster_time), desc='applying NCQE selection cuts to on-beam data...'):

    # 1. Clusterization (bundled with Only Prompt -- see note above)
    c1 = only_prompt_cluster[i]
    if not c1:
        continue
    waterfall_onbeam[0] += 1

    # 2. No FMV/MRD
    c2 = not (TankMRDCoinc[i] == 1 or NoVeto[i] == 0 or MRD_activity[i] != 0 or Veto_activity[i] != 0)
    if not c2:
        continue
    waterfall_onbeam[1] += 1

    # 3. In Spill
    c3 = bunch_time_start_data <= cluster_time[i] <= bunch_time_end_data
    if not c3:
        continue
    waterfall_onbeam[2] += 1

    # 4. Only Prompt -- pass-through (bundled into only_prompt_cluster, see note above)
    waterfall_onbeam[3] += 1

    # 5. q_b
    c5 = cluster_Qb[i] < 0.6
    if not c5:
        continue
    waterfall_onbeam[4] += 1

    # 6. FV
    r2 = Vtx_reco[i]**2 + Vtz_reco[i]**2
    c6 = (Vtx_reco[i] >= -50) and (r2 <= FV_radius**2) and (np.abs(Vty_reco[i]) <= FV_half_height)
    if not c6:
        continue
    waterfall_onbeam[5] += 1

    # 7. E_reco
    recoE = reco_energy(cluster_charge[i])
    c7 = 5 <= recoE <= 12
    if not c7:
        continue
    waterfall_onbeam[6] += 1

    # 8. 1-sigma BNB
    c8 = bunch_cutting(cluster_time[i], centers_data)
    if c8:
        waterfall_onbeam[7] += 1

    # 9. C_V
    cw = pairwise_relative_direction(hitX[i], hitY[i], hitZ[i], hitPE[i], hitT[i])
    c9 = is_inside_rotated_parabola(cw[2], cw[0])
    if c8 and c9:
        waterfall_onbeam[8] += 1

    # 10. 1/3 spill (final)
    c10 = cluster_time[i] <= data_third_spill_cut
    if c8 and c9 and c10:
        waterfall_onbeam[9] += 1


print('done')

applying NCQE selection cuts to on-beam data...: 100%|██████████████████████████████████████████████| 3542081/3542081 [00:05<00:00, 654271.30it/s]

done


In [8]:
# ext data

# Off-beam waterfall, identical structure to the on-beam cell above, against
# the *_ext arrays. Same Clusterization/Only-Prompt bundling limitation.

waterfall_offbeam = np.zeros(len(STAGE_NAMES))

for i in trange(len(cluster_time_ext), desc='applying NCQE selection cuts to off-beam data...'):

    c1 = only_prompt_cluster_ext[i]
    if not c1:
        continue
    waterfall_offbeam[0] += 1

    c2 = not (TankMRDCoinc_ext[i] == 1 or NoVeto_ext[i] == 0 or MRD_activity_ext[i] != 0 or Veto_activity_ext[i] != 0)
    if not c2:
        continue
    waterfall_offbeam[1] += 1

    c3 = bunch_time_start_data <= cluster_time_ext[i] <= bunch_time_end_data
    if not c3:
        continue
    waterfall_offbeam[2] += 1

    waterfall_offbeam[3] += 1   # Only Prompt -- pass-through, see note in on-beam cell

    c5 = cluster_Qb_ext[i] < 0.6
    if not c5:
        continue
    waterfall_offbeam[4] += 1

    r2 = Vtx_reco_ext[i]**2 + Vtz_reco_ext[i]**2
    c6 = (Vtx_reco_ext[i] >= -50) and (r2 <= FV_radius**2) and (np.abs(Vty_reco_ext[i]) <= FV_half_height)
    if not c6:
        continue
    waterfall_offbeam[5] += 1

    recoE = reco_energy(cluster_charge_ext[i])
    c7 = 5 <= recoE <= 12
    if not c7:
        continue
    waterfall_offbeam[6] += 1

    c8 = bunch_cutting(cluster_time_ext[i], centers_data)
    if c8:
        waterfall_offbeam[7] += 1

    cw = pairwise_relative_direction(hitX_ext[i], hitY_ext[i], hitZ_ext[i], hitPE_ext[i], hitT_ext[i])
    c9 = is_inside_rotated_parabola(cw[2], cw[0])
    if c8 and c9:
        waterfall_offbeam[8] += 1

    c10 = cluster_time_ext[i] <= data_third_spill_cut
    if c8 and c9 and c10:
        waterfall_offbeam[9] += 1
        

print('done')

applying NCQE selection cuts to off-beam data...: 100%|███████████████████████████████████████████████| 286189/286189 [00:00<00:00, 942652.03it/s]

done


In [9]:
# add skyshine samples to populate observable plots

# scalar accumulators
skyshine_cluster_number    = []
skyshine_cluster_time      = []
skyshine_cluster_charge    = []
skyshine_cluster_Qb        = []
skyshine_cluster_Hits      = []
skyshine_TankMRDCoinc      = []
skyshine_NoVeto            = []
skyshine_recoVtx           = []
skyshine_recoVty           = []
skyshine_recoVtz           = []
skyshine_event_number      = []

# jagged accumulators
skyshine_hitX  = []
skyshine_hitY  = []
skyshine_hitZ  = []
skyshine_hitT  = []
skyshine_hitPE = []
skyshine_hitID = []
skyshine_MRD_hitT  = []
skyshine_FMV_hitT  = []

for file_name in skyshine_file_names:
    path = os.path.join(file_name)
    root = uproot.open(path)

    # ── cluster tree ──────────────────────────────────────────────────────────
    T   = root['phaseIITankClusterTree']
    CEN = T['eventNumber'].array(library='np')

    skyshine_cluster_number.append(T['clusterNumber'].array(library='np'))
    skyshine_cluster_charge.append(T['clusterPE'].array(library='np'))
    skyshine_cluster_Qb.append(T['clusterChargeBalance'].array(library='np'))
    skyshine_cluster_Hits.append(T['clusterHits'].array(library='np'))
    skyshine_cluster_time.append(T['clusterTime'].array(library='np'))
    skyshine_TankMRDCoinc.append(T['TankMRDCoinc'].array(library='np'))
    skyshine_NoVeto.append(T['NoVeto'].array(library='np'))
    skyshine_recoVtx.append(T['recoLeastSqVtxX'].array(library='np'))
    skyshine_recoVty.append(T['recoLeastSqVtxY'].array(library='np'))
    skyshine_recoVtz.append(T['recoLeastSqVtxZ'].array(library='np'))
    skyshine_event_number.append(CEN)

    skyshine_hitX.append(T['hitX'].array())
    skyshine_hitY.append(T['hitY'].array())
    skyshine_hitZ.append(T['hitZ'].array())
    skyshine_hitT.append(T['hitT'].array())
    skyshine_hitPE.append(T['hitPE'].array())
    skyshine_hitID.append(T['hitDetID'].array())

    # ── trigger tree ──────────────────────────────────────────────────────────
    S   = root['phaseIITriggerTree']

    # map trigger-level quantities onto clusters via CEN
    mrd_t   = S['MRDhitT'].array()
    veto_t  = S['FMVhitT'].array()
    skyshine_MRD_hitT.append(mrd_t[CEN])
    skyshine_FMV_hitT.append(veto_t[CEN])

# ── concatenate everything once ───────────────────────────────────────────────
print('\nManaging arrays...\n')

skyshine_cluster_number  = np.concatenate(skyshine_cluster_number)
skyshine_cluster_time    = np.concatenate(skyshine_cluster_time)
skyshine_cluster_charge  = np.concatenate(skyshine_cluster_charge)
skyshine_cluster_Qb      = np.concatenate(skyshine_cluster_Qb)
skyshine_cluster_Hits    = np.concatenate(skyshine_cluster_Hits)
skyshine_TankMRDCoinc    = np.concatenate(skyshine_TankMRDCoinc)
skyshine_NoVeto          = np.concatenate(skyshine_NoVeto)
skyshine_recoVtx         = np.concatenate(skyshine_recoVtx)
skyshine_recoVty         = np.concatenate(skyshine_recoVty)
skyshine_recoVtz         = np.concatenate(skyshine_recoVtz)
skyshine_event_number    = np.concatenate(skyshine_event_number)

skyshine_hitX     = ak.concatenate(skyshine_hitX)
skyshine_hitY     = ak.concatenate(skyshine_hitY)
skyshine_hitZ     = ak.concatenate(skyshine_hitZ)
skyshine_hitT     = ak.concatenate(skyshine_hitT)
skyshine_hitPE    = ak.concatenate(skyshine_hitPE)
skyshine_hitID    = ak.concatenate(skyshine_hitID)
skyshine_MRD_hitT = ak.concatenate(skyshine_MRD_hitT)
skyshine_FMV_hitT = ak.concatenate(skyshine_FMV_hitT)

print(f'Total clusters: {len(skyshine_cluster_number)}')
print('\ndone')


Managing arrays...

Total clusters: 23216

done


In [10]:
# Skyshine reco-quality-only waterfall: same booleans as the existing skyshine
# selection cell, but WITHOUT its "MC_third_spill_cut" early-exit (that would
# bias these ratios toward only the first 1/3 of the spill) and without any
# timing/bunch-window cuts at all -- those are handled analytically in the
# next cell instead, since the simulated skyshine timing is ad hoc, not
# physical. Gives raw (unweighted) survival counts through the
# reconstruction-quality chain only.

R = dict.fromkeys(['Clusterization', 'No FMV/MRD', 'Only Prompt', 'q_b', 'FV', 'E_reco', 'C_V'], 0)

for i in trange(len(skyshine_cluster_time), desc='skyshine reco-quality waterfall'):

    # 1. Clusterization
    if not (skyshine_cluster_number[i] == 0 and skyshine_cluster_time[i] <= time_to_prompt_end
            and skyshine_cluster_Hits[i] >= 10):
        continue
    R['Clusterization'] += 1

    # 2. No FMV/MRD
    if (
        skyshine_TankMRDCoinc[i] == 1
        or skyshine_NoVeto[i] == 0
        or any(0 <= t <= 300 for t in skyshine_MRD_hitT[i])
        or any(0 <= t <= 300 for t in skyshine_FMV_hitT[i])
    ):
        continue
    R['No FMV/MRD'] += 1

    # 3. Only Prompt
    skip_event = False
    for k in range(i + 1, len(skyshine_cluster_time)):
        if skyshine_cluster_number[k] != 0:
            if skyshine_cluster_time[k] < time_to_prompt_end:
                skip_event = True
            break
        else:
            break
    if skip_event:
        continue
    R['Only Prompt'] += 1

    # 4. q_b
    if skyshine_cluster_Qb[i] >= 0.6:
        continue
    R['q_b'] += 1

    # 5. FV
    r2 = skyshine_recoVtx[i]**2 + skyshine_recoVtz[i]**2
    if skyshine_recoVtx[i] < -50 or r2 > FV_radius**2 or np.abs(skyshine_recoVty[i]) > FV_half_height:
        continue
    R['FV'] += 1

    # 6. E_reco
    recoE = reco_energy(skyshine_cluster_charge[i])
    if recoE < 5 or recoE > 12:
        continue
    R['E_reco'] += 1

    # 7. C_V
    cw = pairwise_relative_direction(skyshine_hitX[i], skyshine_hitY[i], skyshine_hitZ[i], skyshine_hitPE[i], skyshine_hitT[i])
    if not is_inside_rotated_parabola(cw[2], cw[0]):
        continue
    R['C_V'] += 1

print(R)
print('done')

skyshine reco-quality waterfall: 100%|███████████████████████████████████████████████████████████████████| 23216/23216 [00:01<00:00, 18029.91it/s]

{'Clusterization': 22984, 'No FMV/MRD': 22518, 'Only Prompt': 22326, 'q_b': 21846, 'FV': 2374, 'E_reco': 1293, 'C_V': 817}
done


In [11]:
# Skyshine: stitch the analytic sideband time-fit (gives absolute yields at
# specific TIMING extents) together with the raw reco-quality ratios above
# (R, from the cell above). Assumes timing and reconstruction-quality cuts
# are independent for this beam-uncorrelated background -- reasonable, but
# the whole column remains an approximation/extrapolation, as your table
# caption already says.

def calculate_skyshine_in_window(t_cut, t0, err_t0, A, err_A):
    if t_cut <= t0:
        return 0.0, 0.0
    delta_t = t_cut - t0
    N_events = 0.5 * A * (delta_t**2)
    term_A = (0.5 * (delta_t**2) * err_A)**2
    term_t0 = (A * delta_t * err_t0)**2
    err_N = np.sqrt(term_A + term_t0)
    return N_events, err_N


# --- fit parameters from the sideband time fit (04_Skyshine_Sideband.ipynb) ---
T0, ERR_T0 = 353.6, 57.8
A_FIT, ERR_A_FIT = 0.001204, 0.000107

full_spill_width   = bunch_time_end_data - bunch_time_start_data         # ~1521.15 ns
third_spill_width  = data_third_spill_cut - bunch_time_start_data       # ~479.56 ns
prompt_width       = full_spill_width + data_time_to_end_of_buffer      # ~1810.44 ns, out to the ~2us prompt window

N_third,  err_third  = calculate_skyshine_in_window(third_spill_width, T0, ERR_T0, A_FIT, ERR_A_FIT)
N_full,   err_full   = calculate_skyshine_in_window(full_spill_width,  T0, ERR_T0, A_FIT, ERR_A_FIT)
N_prompt, err_prompt = calculate_skyshine_in_window(prompt_width,      T0, ERR_T0, A_FIT, ERR_A_FIT)

A_third = N_third * EXPOSURE_SCALE   # signal-region (1/3-spill, in-bunch) anchor -- validation target ~14.1
A_full  = N_full  * EXPOSURE_SCALE   # full-spill, in-bunch anchor -- validation target ~1210.3

print(f'A_third (validation target {EXPECTED_SKYSHINE}) = {A_third:.2f} +/- {err_third*EXPOSURE_SCALE:.2f}')
print(f'A_full  (validation target ~1210.3)              = {A_full:.2f} +/- {err_full*EXPOSURE_SCALE:.2f}')

InSpill_fraction = N_full / N_prompt                       # EXPOSURE_SCALE cancels in this ratio
bunch_exposure   = bunch_sigma * 2 * len(centers_data)      # 567 ns -- NOT the 182ns from EXPOSURE_SCALE (that's the 26-bunch-specific value, doesn't apply here)

CV_ratio = R['C_V'] / R['E_reco']
S7 = (A_full / CV_ratio) * (full_spill_width / bunch_exposure)   # E_reco stage, in-spill timing scope

skyshine_waterfall = np.array([
    S7 / InSpill_fraction * R['Clusterization'] / R['E_reco'],   # Clusterization
    S7 / InSpill_fraction * R['No FMV/MRD']     / R['E_reco'],   # No FMV/MRD
    S7                    * R['No FMV/MRD']     / R['E_reco'],   # In Spill      (= No FMV/MRD * InSpill_fraction)
    S7                    * R['Only Prompt']    / R['E_reco'],   # Only Prompt
    S7                    * R['q_b']            / R['E_reco'],   # q_b
    S7                    * R['FV']             / R['E_reco'],   # FV
    S7,                                                           # E_reco
    A_full / CV_ratio,                                           # 1-sigma BNB
    A_full,                                                       # C_V
    A_third,                                                       # 1/3 spill (Final)
])

assert np.all(np.diff(skyshine_waterfall) <= 1e-6), 'skyshine cascade is not monotonically non-increasing -- check ratios/anchors'

for name, val in zip(STAGE_NAMES, skyshine_waterfall):
    print(f'{name:16s} {val:10.2f}')
print('done')

A_third (validation target 14.1) = 14.09 +/- 12.99
A_full  (validation target ~1210.3)              = 1210.32 +/- 161.03
Clusterization    142222.05
No FMV/MRD        139338.50
In Spill           89494.82
Only Prompt        88731.74
q_b                86824.04
FV                  9435.15
E_reco              5138.86
1sigma BNB          1915.48
C_V                 1210.32
1/3 spill             14.09
done


In [12]:
# Assemble the waterfall table into a DataFrame and export to CSV.
#
# Reverse-engineered from TABLE_FORMAT.txt (caption text doesn't fully match
# its own numbers -- verified these against several rows by hand):
#   Ratio  = On-beam / Pred
#   eff(%) = 100 * nuNCQE_stage / nuNCQE_true(no-selection)
#   P_MC(%)   = 100 * nuNCQE_stage_scaled / sum(all 6 MC categories, POT+alpha-scaled)   [excludes skyshine, off-beam]
#   P_data(%) = 100 * nuNCQE_stage_scaled / On-beam_stage   [NOT / Pred, despite the caption wording]

POT_MC = (POT/(1e20))

POT_DATA = total_POT_TOR860
TRIG_DATA = total_triggers
TRIG_EXT = total_ext_triggers


pot_scale  = POT_DATA / POT_MC
trig_scale = TRIG_DATA / TRIG_EXT

cat_idx = {'nuNCQE': 0, 'NCother': 1, 'CC': 2, 'nubarNCQE': 3, 'OOFV': 4, 'External': 5}

rows = []

# --- True (no selection) row ---
row = {'Cut': 'True (no selection)', 'On-beam': np.nan, 'Pred': np.nan, 'Ratio': np.nan}
for name, idx in cat_idx.items():
    row[name] = MC_TRUE_by_cat[idx]   # raw MC-POT truth counts, unscaled -- matches table convention
row['Skyshine'] = np.nan
row['Offbeam']  = np.nan
row['eff']      = np.nan
row['P_MC']     = np.nan
row['P_data']   = np.nan
rows.append(row)

# --- selection stages ---
for s, stage in enumerate(STAGE_NAMES):
    row = {'Cut': stage, 'On-beam': waterfall_onbeam[s]}

    mc_scaled = {}
    for name, idx in cat_idx.items():
        val = pot_scale * MC_waterfall[s, idx]
        if name == 'External':
            val *= ALPHA_SCALING
        mc_scaled[name] = val
        row[name] = val

    row['Offbeam']  = trig_scale * waterfall_offbeam[s]
    row['Skyshine'] = skyshine_waterfall[s]

    mc_total = sum(mc_scaled.values())                        # excludes skyshine/off-beam -- P_MC denominator
    pred     = mc_total + row['Offbeam'] + row['Skyshine']     # NaN-propagates while Skyshine is NaN
    row['Pred']  = pred
    row['Ratio'] = row['On-beam'] / pred

    row['eff']    = 100 * MC_waterfall[s, cat_idx['nuNCQE']] / MC_TRUE_by_cat[cat_idx['nuNCQE']]
    row['P_MC']   = 100 * mc_scaled['nuNCQE'] / mc_total
    row['P_data'] = 100 * mc_scaled['nuNCQE'] / row['On-beam']

    rows.append(row)

col_order = ['Cut', 'On-beam', 'Pred', 'Ratio', 'nuNCQE', 'NCother', 'CC',
             'nubarNCQE', 'External', 'OOFV', 'Skyshine', 'Offbeam',
             'eff', 'P_MC', 'P_data']
waterfall_df = pd.DataFrame(rows)[col_order].rename(columns={'External': 'Ext.'})

csv_path = os.path.join(plots_dir, 'NCQE_waterfall_counts_Base_GENIE.csv')
waterfall_df.to_csv(csv_path, index=False)
print(f'Wrote {csv_path}')
waterfall_df

Wrote plots/NCQE_waterfall_counts_Base_GENIE.csv


,Cut,On-beam,Pred,Ratio,nuNCQE,NCother,CC,nubarNCQE,Ext.,OOFV,Skyshine,Offbeam,eff,P_MC,P_data
0,True (no selection),NaN,NaN,NaN,19553.000000,22427.000000,114970.430125,349.000000,4.907390e+06,1.184485e+06,NaN,NaN,NaN,NaN,NaN
1,Clusterization,779335.0,967497.792119,0.805516,7560.208438,10665.096809,66433.594988,137.194094,1.550700e+05,4.760532e+05,142222.048789,109356.457435,66.511533,1.056014,0.970085
2,No FMV/MRD,556801.0,729081.706825,0.763702,7132.930221,8353.143794,37632.857128,133.124778,1.181360e+05,3.089987e+05,139338.500463,109356.457435,62.752519,1.484831,1.281056
3,In Spill,477087.0,644679.463946,0.740038,7092.818389,8308.962645,37391.861344,132.543447,1.162105e+05,3.065095e+05,89494.819470,79538.430058,62.399632,1.491196,1.486693
4,Only Prompt,477087.0,571536.317012,0.834745,7040.498607,7814.250044,27868.917186,132.543447,9.882216e+04,2.615878e+05,88731.740807,79538.430058,61.939344,1.745869,1.475726
5,q_b,466324.0,562385.641403,0.829189,6989.922818,7780.532851,27809.725027,131.380785,9.825797e+04,2.573230e+05,86824.044149,77269.054423,61.494400,1.754972,1.498941
6,FV,45304.0,49588.956839,0.913591,2481.701639,3204.295956,10332.348533,56.970429,3.518728e+03,1.333746e+04,9435.149721,7222.300212,21.832967,7.535949,5.477886
7,E_reco,11978.0,13022.492892,0.919793,1464.372553,402.280987,109.769329,36.042516,1.439105e+03,3.165038e+03,5138.857873,1267.027217,12.882934,22.131772,12.225518
8,1sigma BNB,5291.0,6046.903716,0.874993,1030.699697,258.110922,71.506523,27.322553,5.730639e+02,1.695275e+03,1915.483116,475.441548,9.067662,28.192166,19.480244
9,C_V,3385.0,3593.885049,0.941878,635.976012,158.703337,42.096717,20.927913,2.617738e+02,9.650936e+02,1210.324598,298.989015,5.595049,30.508718,18.788065


In [13]:
# Re-create the LaTeX table in the same structure as descr/TABLE_FORMAT.txt.
# NOTE: the caption prose below is a minimal placeholder with the numeric
# values filled in dynamically -- rewrite the descriptive text yourself
# before this goes in the thesis/paper, I haven't attempted to reproduce your
# prose style.

def build_latex_table(df, pot_data_e20, out_path):

    def fmt_num(x, sci_thresh=1e6):
        if pd.isna(x):
            return '---'
        if abs(x) >= sci_thresh:
            exp = int(np.floor(np.log10(abs(x))))
            mant = x / 10**exp
            return fr'${mant:.3f}\times10^{{{exp}}}$'
        return f'{x:.1f}'

    def fmt_pct(x):
        return '---' if pd.isna(x) else f'{x:.2f}'

    def row_line(cut, on_beam, pred, ratio, nuncqe, ncother, cc, nubar, ext, oofv,
                 sky, offb, eff, pmc, pdata, bold=False):
        vals = [cut, fmt_num(on_beam), fmt_num(pred), fmt_pct(ratio),
                fmt_num(nuncqe), fmt_num(ncother), fmt_num(cc), fmt_num(nubar),
                fmt_num(ext), fmt_num(oofv), fmt_num(sky), fmt_num(offb),
                fmt_pct(eff), fmt_pct(pmc), fmt_pct(pdata)]
        if bold:
            vals = [vals[0]] + [fr'\textbf{{{v}}}' for v in vals[1:]]
        return '      & '.join(vals) + r' \\'

    true_row  = df.iloc[0]
    presel    = df.iloc[1:6]     # Clusterization .. q_b
    sigsel    = df.iloc[6:11]    # FV .. 1/3 spill
    final_row = df.iloc[-1]      # 1/3 spill == Final signal

    lines = []
    lines.append(r'\begin{sidewaystable}[htbp]')
    lines.append(r'  \centering')
    lines.append(r'  \isucaption[\texorpdfstring{$\nu$}{ν}NCQE event selection summary]{%')
    lines.append(
        fr'    Summary of event yields at each stage of the $\nu$NCQE selection for '
        fr'${pot_data_e20:.2f}\times10^{{20}}$~POT. External (dirt) events scaled by '
        fr'$\alpha_{{\text{{dirt}}}} = {ALPHA_SCALING}$. TODO: rewrite this caption -- '
        fr'skyshine values prior to the final signal window are placeholders (NaN) pending '
        fr'your sideband extrapolation.'
    )
    lines.append(r'  }')
    lines.append(r'  \label{tab:NCQE_event_selection}')
    lines.append(r'  \scriptsize')
    lines.append(r'  \setlength{\tabcolsep}{3pt}')
    lines.append(r'  \renewcommand{\arraystretch}{1.25}')
    lines.append(r'  \tagpdfsetup{table/header-rows={1}}')
    lines.append(r'  \begin{tabular}{l r r c r r r r r r r r r r r}')
    lines.append(r'    \toprule')
    lines.append('')
    header_cells = [r'\textbf{Cut}', r'\textbf{On-beam}', r'\textbf{Pred.}', r'\textbf{Ratio}',
                     r'\textbf{$\nu$NCQE}', r'\textbf{NCother}', r'\textbf{CC}',
                     r'\textbf{$\bar{\nu}$NCQE}', r'\textbf{Ext.}', r'\textbf{OOFV}',
                     r'\textbf{Skyshine}', r'\textbf{Off-beam}', r'\textbf{$\epsilon$ (\%)}',
                     r'\textbf{$P_{\text{MC}}$ (\%)}', r'\textbf{$P_{\text{data}}$ (\%)}']
    lines.append('    ' + header_cells[0])
    for h in header_cells[1:-1]:
        lines.append(f'      & {h}')
    lines.append(f'      & {header_cells[-1]} \\\\')
    lines.append('')
    lines.append(r'    \midrule')
    lines.append(r'    \multicolumn{15}{l}{\textit{\small MC true event rates (no selection applied)}} \\')
    lines.append(r'    \cmidrule(lr){1-15}')
    lines.append('')
    lines.append(row_line('---', true_row['On-beam'], true_row['Pred'], true_row['Ratio'],
                           true_row['nuNCQE'], true_row['NCother'], true_row['CC'],
                           true_row['nubarNCQE'], true_row['Ext.'], true_row['OOFV'],
                           true_row['Skyshine'], true_row['Offbeam'],
                           true_row['eff'], true_row['P_MC'], true_row['P_data']))
    lines.append('')
    lines.append(r'    \midrule')
    lines.append(r'    \multicolumn{15}{l}{\textit{\small Pre-selection}} \\')
    lines.append(r'    \cmidrule(lr){1-15}')
    lines.append('')
    for _, r in presel.iterrows():
        lines.append(row_line(r['Cut'], r['On-beam'], r['Pred'], r['Ratio'],
                               r['nuNCQE'], r['NCother'], r['CC'], r['nubarNCQE'],
                               r['Ext.'], r['OOFV'], r['Skyshine'], r['Offbeam'],
                               r['eff'], r['P_MC'], r['P_data']))
        lines.append('')
    lines.append(r'    \midrule')
    lines.append(r'    \multicolumn{15}{l}{\textit{\small Signal selection}} \\')
    lines.append(r'    \cmidrule(lr){1-15}')
    lines.append('')
    for _, r in sigsel.iterrows():
        lines.append(row_line(r['Cut'], r['On-beam'], r['Pred'], r['Ratio'],
                               r['nuNCQE'], r['NCother'], r['CC'], r['nubarNCQE'],
                               r['Ext.'], r['OOFV'], r['Skyshine'], r['Offbeam'],
                               r['eff'], r['P_MC'], r['P_data']))
        lines.append('')
    lines.append(r'    \midrule')
    lines.append('')
    lines.append(row_line('Final signal', final_row['On-beam'], final_row['Pred'], final_row['Ratio'],
                           final_row['nuNCQE'], final_row['NCother'], final_row['CC'],
                           final_row['nubarNCQE'], final_row['Ext.'], final_row['OOFV'],
                           final_row['Skyshine'], final_row['Offbeam'],
                           final_row['eff'], final_row['P_MC'], final_row['P_data'], bold=True))
    lines.append('')
    lines.append('')
    lines.append(r'    \bottomrule')
    lines.append(r'  \end{tabular}')
    lines.append(r'\end{sidewaystable}')

    with open(out_path, 'w') as f:
        f.write('\n'.join(lines) + '\n')
    print(f'Wrote {out_path}')


tex_path = os.path.join(plots_dir, 'NCQE_waterfall_table_Base_GENIE.tex')
build_latex_table(waterfall_df, POT_DATA, tex_path)

Wrote plots/NCQE_waterfall_table_Base_GENIE.tex


In [14]:
# Waterfall efficiency/purity plot, reading straight from waterfall_df
# instead of hardcoded lists. Uses MultipleLocator directly since that's
# what's imported by name in lib/imports.py (not `matplotlib.ticker as ticker`).

display_names = {
    'Clusterization': 'Clusterization',
    'No FMV/MRD': 'No MRD/Veto',
    'In Spill': 'In Spill',
    'Only Prompt': 'Only Prompt',
    'q_b': 'Charge Balance',
    'FV': 'FV',
    'E_reco': r'$E_\mathrm{reco}$',
    '1sigma BNB': r'$1\sigma$ BNB',
    'C_V': r'$C_V$',
    '1/3 spill': 'Final',
}
categories = ['No Selection'] + [display_names[s] for s in STAGE_NAMES]

true_mc_purity = 100 * MC_TRUE_by_cat[cat_idx['nuNCQE']] / MC_TRUE_by_cat.sum()

eff      = [100.0] + waterfall_df['eff'].iloc[1:].tolist()
pur_mc   = [true_mc_purity] + waterfall_df['P_MC'].iloc[1:].tolist()
pur_data = [np.nan] + waterfall_df['P_data'].iloc[1:].tolist()

x = np.arange(len(categories))
pur_data_arr = np.array(pur_data, dtype=float)

fig, ax1 = plt.subplots(figsize=(9, 5))

color_eff = '#185FA5'
l1, = ax1.plot(x, eff, color=color_eff, marker='o', linewidth=2,
               markersize=6, markerfacecolor=color_eff,
               markeredgecolor='white', markeredgewidth=1.2,
               label='Efficiency', zorder=3)

ax1.set_ylabel('Signal Efficiency (%)', color=color_eff, fontsize=14)
ax1.tick_params(axis='y', labelcolor=color_eff)
ax1.set_ylim(-2, 105)
ax1.yaxis.set_major_locator(MultipleLocator(10))

ax2 = ax1.twinx()

color_mc   = '#0F6E56'
color_data = '#BA7517'

l2, = ax2.plot(x, pur_mc, color=color_mc, marker='o', linewidth=2,
               markersize=6, markerfacecolor=color_mc,
               markeredgecolor='white', markeredgewidth=1.2,
               label='Purity (MC)', zorder=3)

l3, = ax2.plot(x, pur_data_arr, color=color_data, marker='o', linewidth=2,
               linestyle='--', markersize=6, markerfacecolor=color_data,
               markeredgecolor='white', markeredgewidth=1.2,
               label='Purity (Data)', zorder=3)

ax2.set_ylabel('Signal Purity (%)', fontsize=14)
ax2.tick_params(axis='y')
ax2.set_ylim(-2, max(42, np.nanmax(pur_mc) * 1.15))   # adjust once real numbers are in
ax2.yaxis.set_major_locator(MultipleLocator(5))

ax1.set_xticks(x)
ax1.set_xticklabels(categories, rotation=38, ha='right', fontsize=12)
ax1.set_xlim(-0.3, len(categories) - 0.7)

ax1.grid(True, linestyle='--', alpha=0.4, zorder=0)
ax1.set_axisbelow(True)

lines  = [l1, l2, l3]
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, fontsize=12, frameon=True,
           framealpha=0.9, loc='upper center', ncol=3)

ax1.set_title(r'$\nu$NCQE Event Selection Waterfall', fontsize=15, pad=10)

plt.tight_layout()
plt.savefig(os.path.join(plots_dir, 'NCQE_waterfall_BASE_GENIE.png'), dpi=300,
            bbox_inches='tight', pad_inches=0.3, facecolor='w')
plt.show()

print('done')

done


In [8]:
# Final comparison cell: overlay Base GENIE vs Empirical model waterfall
# curves (efficiency + MC/data purity) on one plot. Reads directly from the
# two exported CSVs, so this doesn't depend on which MC model is currently
# loaded in the kernel -- run it any time after both CSVs exist on disk.

empirical_df = pd.read_csv(os.path.join(plots_dir, 'NCQE_waterfall_counts.csv'))
base_df      = pd.read_csv(os.path.join(plots_dir, 'NCQE_waterfall_counts_Base_GENIE.csv'))

STAGE_NAMES = ['Clusterization', 'No FMV/MRD', 'In Spill', 'Only Prompt', 'q_b',
               'FV', 'E_reco', '1sigma BNB', 'C_V', '1/3 spill']
display_names = {
    'Clusterization': 'Clusterization',
    'No FMV/MRD': 'No MRD/Veto',
    'In Spill': 'In Spill',
    'Only Prompt': 'Only Prompt',
    'q_b': 'Charge Balance',
    'FV': 'FV',
    'E_reco': r'$E_\mathrm{reco}$',
    '1sigma BNB': r'$1\sigma$ BNB',
    'C_V': r'$C_V$',
    '1/3 spill': 'Final',
}
categories = ['No Selection'] + [display_names[s] for s in STAGE_NAMES]


def true_mc_purity(df):
    true_row = df.iloc[0]
    cats = ['nuNCQE', 'NCother', 'CC', 'nubarNCQE', 'Ext.', 'OOFV']
    return 100 * true_row['nuNCQE'] / true_row[cats].sum()


def build_curves(df):
    stages = df.iloc[1:]   # drop the "True (no selection)" row
    eff      = [100.0] + stages['eff'].tolist()
    pur_mc   = [true_mc_purity(df)] + stages['P_MC'].tolist()
    pur_data = [np.nan] + stages['P_data'].tolist()
    return eff, pur_mc, np.array(pur_data, dtype=float)


eff_emp,  pmc_emp,  pdata_emp  = build_curves(empirical_df)
eff_base, pmc_base, pdata_base = build_curves(base_df)

x = np.arange(len(categories))

fig, ax1 = plt.subplots(figsize=(10, 5.5))

color_eff  = '#185FA5'
color_mc   = '#0F6E56'
color_data = '#BA7517'

# --- Empirical model: filled circles, solid lines ---
l1, = ax1.plot(x, eff_emp, color=color_eff, marker='o', linewidth=2,
               markersize=6, markerfacecolor=color_eff,
               markeredgecolor='white', markeredgewidth=1.2,
               linestyle='-', label='Efficiency', zorder=3)

ax2 = ax1.twinx()
l2, = ax2.plot(x, pmc_emp, color=color_mc, marker='o', linewidth=2,
               markersize=6, markerfacecolor=color_mc,
               markeredgecolor='white', markeredgewidth=1.2,
               linestyle='-', label='Purity (MC)', zorder=3)
l3, = ax2.plot(x, pdata_emp, color=color_data, marker='o', linewidth=2,
               markersize=6, markerfacecolor=color_data,
               markeredgecolor='white', markeredgewidth=1.2,
               linestyle='--', label='Purity (Data)', zorder=3)

# --- Base GENIE: hollow squares, dotted lines ---
l4, = ax1.plot(x, eff_base, color=color_eff, marker='o', linewidth=2,
               markersize=6, markerfacecolor='white',
               markeredgecolor=color_eff, markeredgewidth=1.4,
               linestyle=':', label='Efficiency (Base GENIE)', zorder=3)
l5, = ax2.plot(x, pmc_base, color=color_mc, marker='o', linewidth=2,
               markersize=6, markerfacecolor='white',
               markeredgecolor=color_mc, markeredgewidth=1.4,
               linestyle=':', label='Purity (MC, Base GENIE)', zorder=3)
l6, = ax2.plot(x, pdata_base, color=color_data, marker='o', linewidth=2,
               markersize=6, markerfacecolor='white',
               markeredgecolor=color_data, markeredgewidth=1.4,
               linestyle=':', label='Purity (Data, Base GENIE)', zorder=3)

ax1.set_ylabel('Signal Efficiency (%)', color=color_eff, fontsize=14)
ax1.tick_params(axis='y', labelcolor=color_eff)
ax1.set_ylim(-2, 105)
ax1.yaxis.set_major_locator(MultipleLocator(10))

ax2.set_ylabel('Signal Purity (%)', fontsize=14)
ax2.tick_params(axis='y')
ax2.set_ylim(-2, max(42, np.nanmax(pmc_emp) * 1.15, np.nanmax(pmc_base) * 1.15))
ax2.yaxis.set_major_locator(MultipleLocator(5))

ax1.set_xticks(x)
ax1.set_xticklabels(categories, rotation=38, ha='right', fontsize=12)
ax1.set_xlim(-0.3, len(categories) - 0.7)

ax1.grid(True, linestyle='--', alpha=0.4, zorder=0)
ax1.set_axisbelow(True)

# --- single legend, 6 handles / ncol=3 -> wraps into 2 rows automatically:
# row 1 = Empirical (implied default), row 2 = explicitly labeled Base GENIE ---
#handles = [l1, l2, l3, l4, l5, l6]
#labels  = ['Efficiency', 'Purity (MC)', 'Purity (Data)',
#           'Efficiency (Base GENIE)', 'Purity (MC, Base GENIE)', 'Purity (Data, Base GENIE)']

handles = [l1, l4, 
           l2, l5,
           l3, l6]
labels  = ['Efficiency', 'Efficiency (Base GENIE)',
           'Purity (MC)', 'Purity (MC, Base GENIE)', 
           'Purity (Data)', 'Purity (Data, Base GENIE)']

ax1.legend(handles, labels, fontsize=10, frameon=True, framealpha=0.9,
           loc='upper center', ncol=3)

ax1.set_title(r'$\nu$NCQE Event Selection Waterfall: Empirical vs. Base GENIE', fontsize=15, pad=10)

plt.tight_layout()
plt.savefig(os.path.join(plots_dir, 'NCQE_waterfall_comparison.png'), dpi=300,
            bbox_inches='tight', pad_inches=0.3, facecolor='w')
plt.show()
print('done')

done
